In [1]:
from mpi4py import MPI
import coqui

# Create CoQui MPI handler and set logging verbosity in the beginning
coqui_mpi = coqui.MpiHandler()
coqui.set_verbosity(coqui_mpi, output_level=1)

--------------------------------------------------------------------------
Ignoring value for oob_tcp_if_exclude on worker6102 (10.250.112.0/20: Did not find interface matching this subnet).
(You can safely ignore this message.)
--------------------------------------------------------------------------


## From DFT to CoQui

<figure style="text-align: center;">
 <img src="../images/coqui_workflow_wan90.png" alt="Workflow of CoQuí" width="60%">
</figure>

This notebook covers the entry point of a CoQuí many-body workflow: preparing reusable inputs from DFT results.

#### 🔹 Two ingredients CoQuí needs

1. Crystal metadata (k-mesh, lattice, pseudopotentials, unit-cell information).
2. Single-particle Bloch orbitals used as the basis of the interacting many-electron problem. In practice, orbitals usually come from a mean-field method such as DFT, HF, or related single-particle approaches.


> Note: You can also start from a pre-optimized localized basis (for example, Gaussian-type orbitals), as long as the metadata and orbitals are provided in a supported format.

#### 🔹 Why MLWFs appear in this workflow

A selected subset of Bloch orbitals is often used to span maximally localized Wannier functions (MLWFs) [[1](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.65.035109), [2](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.56.12847)].
- MLWFs provide a compact real-space representation for interpolation on dense k-meshes.
- MLWFs define physically motivated subspaces for embedding methods such as DMFT/EDMFT.

#### 🔹 What this notebook covers

1. Convert QE outputs into CoQuí format with `pw2coqui.x`.
2. Declare the physical system through `coqui.Mf` (CoQuí's mean-field data container).
3. Construct MLWFs through CoQuí’s Wannier90 interface.

Reference material:
- QE tutorials: [official tutorials](https://www.quantum-espresso.org/tutorials/)
- Wannier90 references: [documentation](https://wannier90.readthedocs.io/en/latest/) and [tutorial slides](https://docs.epw-code.org/_downloads/f2f39ac545ca12dc2838f7359b3633a9/Mon.4.Pizzi.pdf)


### Section 1: Quantum ESPRESSO converter

<figure style="text-align: center;">
 <img src="../images/coqui_dft_converter.png" alt="Workflow of input preparation for CoQuí" width="60%">
</figure>

Electronic-structure packages such as QE, VASP, and PySCF write their results in different formats. CoQuí uses converter interfaces to translate those outputs into a unified input format for later many-body steps.

This section focuses on the Quantum ESPRESSO path and the QE converter [ `pw2coqui.x` ](https://github.com/AbInitioQHub/coqui/tree/main/qe_converter).

#### 🔹 Converter command

The QE converter is used from the command line:
```bash
pw2coqui.x -in {prefix}.pw2coqui.in
```

Its minimal input file is:
```text
&input_pw2coqui
 prefix = "{prefix}"
 outdir = "{outdir}"
/
```
- `prefix` (required): QE prefix that identifies the dataset and output file names.
- `outdir` (required): QE output directory containing `{prefix}.save/`.

Running the converter generates:
- **`{prefix}.coqui.h5`**: standardized HDF5 metadata for CoQuí (k-point mesh, lattice vectors, pseudopotential information, etc.).

CoQuí also needs access to QE wavefunctions from the `nscf` step:
- **`{outdir}/{prefix}.save/`**: directory containing Kohn–Sham orbitals (`wfc*.hdf5`) used as the Bloch basis.

Together, these form the complete input for subsequent CoQuí steps.


> Important: Do not move or delete `{prefix}.save/` after conversion. Later steps require both `.coqui.h5` and `{prefix}.save/`.


### Section 2: Declaring a simulated physical system

<figure style="text-align: center;">
 <img src="../images/coqui_workflow_mf.png" alt="Workflow of CoQuí's Mf declaration step" width="60%">
 <figcaption><em>Figure&nbsp;2:</em> The mean-field declaration step in the CoQuí workflow.</figcaption>
</figure>

A universal starting point for CoQui simulations is declaring the simulated physical system. In CoQui, a crystal system is defined by its metadata (lattice, k-mesh, pseudopotentials, etc.) and the single-particle orbitals $\phi^{\mathbf{k}}_{i}(\mathbf{r})$, which together determine the non-interacting Hamiltonian: 
$$
(H_{0})^{\textbf{k}}_{ij} = \int d\textbf{r} \, \phi^{\textbf{k}*}_{i}(\textbf{r}) \Big [ \frac{\nabla^{2}}{2} + V_{\mathrm{ext}}(\textbf{r}) \Big ] \phi^{\textbf{k}}_{j}(\textbf{r}),
$$

#### 🔹 `coqui.Mf` container

In CoQuí, the declaration of the simulated system is handled by the read-only `coqui.Mf` container, constructed through:
```python
coqui.make_mf(mpi: coqui.MpiHandler, params: dict, mf_type: str) -> coqui.Mf
```

Function inputs:
- `mpi` (required): `MpiHandler` instance that carries the MPI context.
- `params` (required): dictionary containing input keys.
- `mf_type` (required): backend label, currently `"qe"` or `"pyscf"`.

The returned `coqui.Mf` object is agnostic to external DFT backends (QE, PySCF, etc.) and becomes the common input object for later CoQuí routines.

#### 🔹 Example and key parameters

In [ ]:
# Mf for the target system
params = {
 "prefix": "svo",                           # QE prefix (matches {prefix}.save)
 "outdir": "../data/qe_inputs/svo/222/out", # QE outdir containing {prefix}.save/
 "nbnd": 40                                 # number of bands imported from QE outputs
}
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

The example above constructs an `Mf` object from a QE calculation for SrVO$_3$.

Key `params` dictionary entries for `make_mf(..., mf_type="qe")`:
- `prefix` (required): QE prefix used in `scf/nscf`.
- `outdir` (required): QE outdir containing `{prefix}.save/`.
- `nbnd` (optional): number of imported bands. If omitted, all available bands are imported.

> Tip: Manipulate `nbnd` to control the number of KS orbitals for basis convergence tests and faster experimental runs. 


#### 🔹 Hands-on — `Mf` container for SrVO$_3$ from Quantum ESPRESSO

Create an `Mf` object for SrVO$_3$ using the example above, then complete the checks below.

1. Describe the system from the log output: number of KS orbitals, k-points, spins, and related metadata.
2. Cross-check with QE [input](../data/qe_inputs/svo/222/out/svo.nscf.in)/[output](../data/qe_inputs/svo/222/out/svo.nscf.out) and verify that the `Mf` metadata matches the QE setup.
3. Experiment with `"nbnd"`:
 - Remove `"nbnd"` and rebuild `Mf` to include all KS orbitals. What is the total count?
 - Reintroduce `"nbnd"` with different values and verify the reported number of included orbitals.

In [3]:
# Mf for the target system
params = {
  "prefix": "svo",                        # QE prefix (matches {prefix}.save)
  "outdir": "../data/qe_inputs/svo/222/out", # QE outdir containing {prefix}.save/
  "nbnd": 40                              # number of bands read from QE outputs
}
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

# remove "nbnd"
del params["nbnd"]
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

# set "nbnd" = 5
params["nbnd"] = 5
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

  Quantum ESPRESSO reader
  -----------------------
  Number of spins                = 1
  Number of polarizations        = 1
  Number of bands                = 40
  Monkhorst-Pack mesh            = (2,2,2)
  K-points                       = 8 total, 4 in the IBZ
  Number of electrons            = 41.0
  Electron density energy cutoff = 360.000 a.u. | FFT mesh = (45,45,45)
  Wavefunction energy cutoff     = 51.704 a.u. | FFT mesh = (23,23,23), Number of PWs = 6859

  Quantum ESPRESSO reader
  -----------------------
  Number of spins                = 1
  Number of polarizations        = 1
  Number of bands                = 100
  Monkhorst-Pack mesh            = (2,2,2)
  K-points                       = 8 total, 4 in the IBZ
  Number of electrons            = 41.0
  Electron density energy cutoff = 360.000 a.u. | FFT mesh = (45,45,45)
  Wavefunction energy cutoff     = 51.704 a.u. | FFT mesh = (23,23,23), Number of PWs = 6859

  Quantum ESPRESSO reader
  -----------------------
  Numbe

### Section 3: MLWFs from CoQuí

<figure style="text-align: center;">
 <img src="../images/coqui_workflow_wan90.png" alt="Workflow of CoQuí's Wannier90 interface" width="60%">
 <figcaption><em>Figure&nbsp;2:</em> MLWF construction step in the CoQuí workflow.</figcaption>
</figure>

Maximally localized Wannier functions (MLWFs) [[1](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.65.035109), [2](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.56.12847)] are a key bridge between different CoQuí components.

They are especially useful for:
- interpolation on dense k-meshes;
- defining physically motivated correlated subspaces for DMFT/EDMFT-style embedding.

This section focuses on calling Wannier90 from inside CoQuí so that Wannierization becomes part of the same Python workflow.

#### 🔹 API snapshot

CoQuí exposes the Wannier90 interface through:
```python
coqui.wannier90(mf: coqui.Mf, params: dict)
```

Function inputs:
- `mf` (required): declared physical system (`coqui.Mf`) that provides structure, k-mesh, and Bloch orbitals.
- `params` (required): Wannier90 parameter dictionary.

In this tutorial, the main `params` key is:
- `"prefix"` (required): Wannier90 seedname used to locate `{prefix}.win` and write outputs.


#### 🔹 Example and key parameters

The typical sequence is to build `Mf` first, then call `coqui.wannier90` with a prepared `{prefix}.win` file:

In [ ]:
from mpi4py import MPI
import coqui

# Initialize CoQuí MPI handler
coqui_mpi = coqui.MpiHandler()
coqui.set_verbosity(coqui_mpi, output_level=1)

# Step 1: Build mean-field object from QE outputs
mf_params = {
 "prefix": "svo",
 "outdir": "../data/qe_inputs/svo/222/out",
 "nbnd": 40
}
mf = coqui.make_mf(coqui_mpi, params=mf_params, mf_type="qe")

# Step 2: Run Wannier90 through CoQuí
w90_params = {
    "prefix": "svo",
    "h5_filename": "svo.mlwf.h5"
}
coqui.wannier90(mf=mf, params=w90_params)

Key `params` dictionary entries for `coqui.wannier90(..., param)`:
- `prefix` (required): Wannier90 seedname used to locate `{prefix}.win` and write outputs.
- `h5_filename` (optional): Name of the HDF5 file to store the MLWF data.

> Importantly, `{prefix}.win` should contain Wannierization-specific settings only; structural and k-point data are taken from `coqui.Mf`.

#### 🔹 Hands-on — MLWFs of SrVO$_{3}$

Goal: construct MLWFs for SrVO$_3$ using CoQuí's Wannier90 interface.

Step 1: Copy the `.win` file into the working directory by running the next code cell.


In [4]:
%%bash
cp ../data/qe_inputs/svo/222/mlwf/svo.win .

Step 2: Run the following example to call Wannier90 through CoQuí.

```python
# Step 1: Build mean-field object from existing QE outputs
params = {
 "prefix": "svo",
 "outdir": "../data/qe_inputs/svo/222/out",
 "nbnd": 40
}
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

# Step 2: Run Wannier90 through CoQuí
w90_params = {"prefix": "svo"}
coqui.wannier90(mf=mf, params=w90_params)
```

Questions:
1. How many MLWFs are generated?
2. What are their spreads and centers?


In [ ]:
# Step 1: Build mean-field object from existing QE outputs
params = {
  "prefix": "svo",                    # QE prefix (matches {prefix}.save)
  "outdir": "../data/qe_inputs/svo/222/out", # QE outdir containing {prefix}.save/
  "nbnd": 40                           # number of bands read from QE outputs
}
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

# Step 2: Run Wannier90 through CoQuí
w90_params = {
  "prefix": "svo",     # equivalent to wannier90 seedname
}
coqui.wannier90(mf=mf, params=w90_params)

  Quantum ESPRESSO reader
  -----------------------
  Number of spins                = 1
  Number of polarizations        = 1
  Number of bands                = 40
  Monkhorst-Pack mesh            = (2,2,2)
  K-points                       = 8 total, 4 in the IBZ
  Number of electrons            = 41.0
  Electron density energy cutoff = 360.000 a.u. | FFT mesh = (45,45,45)
  Wavefunction energy cutoff     = 51.704 a.u. | FFT mesh = (23,23,23), Number of PWs = 6859

*************************************************
       Running Wannier90 in library-mode         
*************************************************


 *---------------------------------- K-MESH ----------------------------------*
 +----------------------------------------------------------------------------+
 |                    Distance to Nearest-Neighbour Shells                    |
 |                    ------------------------------------                    |
 |          Shell             Distance (Ang^-1)          M

#### Step 3

Run the next cell to clean up temporary Wannier90 outputs.


In [6]:
%%bash
# clean up Wannier90 outputs
if ls svo_* 1> /dev/null 2>&1; then
    rm svo_*
fi

#### 🔹 Hands-on extension — larger energy window

Continue the Section 3 hands-on by modifying `svo.win` to build MLWFs for both V $d$ and O $p$ states using a larger disentanglement window.

Steps:
1. Edit `svo.win` to include O $p$ bands and the full V $d$ shell.
2. Run `coqui.wannier90` again to generate updated MLWFs.
3. Compare against the small-window result from Hands-on 2a.

Checklist:
- MLWF spreads should become noticeably smaller than in the small-window case.
- Each oxygen atom should have three localized $p$-like orbitals.

Reference: oxygen positions are listed in `data/qe_inputs/svo/222/out/svo.nscf.out`.


In [8]:
%%bash
cp ../data/qe_inputs/svo/222/mlwf_dp/svo.win .

In [ ]:
# Step 1: Build mean-field object from existing QE outputs
params = {
  "prefix": "svo",                    # QE prefix (matches {prefix}.save)
  "outdir": "../data/qe_inputs/svo/222/out", # QE outdir containing {prefix}.save/
  "nbnd": 40                           # number of bands read from QE outputs
}
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

# Step 2: Run Wannier90 through CoQuí
w90_params = {
  "prefix": "svo",     # equivalent to wannier90 seedname
}
coqui.wannier90(mf=mf, params=w90_params)

  Quantum ESPRESSO reader
  -----------------------
  Number of spins                = 1
  Number of polarizations        = 1
  Number of bands                = 40
  Monkhorst-Pack mesh            = (2,2,2)
  K-points                       = 8 total, 4 in the IBZ
  Number of electrons            = 41.0
  Electron density energy cutoff = 360.000 a.u. | FFT mesh = (45,45,45)
  Wavefunction energy cutoff     = 51.704 a.u. | FFT mesh = (23,23,23), Number of PWs = 6859

*************************************************
       Running Wannier90 in library-mode         
*************************************************


 *---------------------------------- K-MESH ----------------------------------*
 +----------------------------------------------------------------------------+
 |                    Distance to Nearest-Neighbour Shells                    |
 |                    ------------------------------------                    |
 |          Shell             Distance (Ang^-1)          M

In [10]:
%%bash
# clean up Wannier90 outputs
if ls svo_* 1> /dev/null 2>&1; then
    rm svo_*
fi